# Dashboard statistics: precomputed tests

A few statistics on the dashboard need SciPy or scikit-learn: the chi-square tests of missing values and of brewery
types, the Spearman correlations between a city's distance from its state's center and its share of the state, the
Ward grouping of states by type mix (with silhouette scores), and the permutation test of same-type neighbours.

Importing SciPy and scikit-learn costs the dashboard about 80 MB of memory in every server process, which matters on a
512 MB host. Every one of these statistics depends only on the data and on a small, fixed set of page settings, so we
compute them here for every setting the pages offer and store them in `data/dashboard_stats.db`. The pages read the
stored tables; the running app no longer imports either library.

**When to rerun:** after any change to the three dataset databases, `data/coordinate_corrections.csv` or
`data/reference_places.db`, so after `brewery_clustering_all_runs.ipynb`. It takes about a minute.
`tests/test_stats_store.py` fails if the stored tables no longer match a fresh computation or if an input file
changed since the build.

In [1]:
import sqlite3

import pandas as pd

import distribution_core as dc
import origins_core as oc
import quality_core as qc
import stats_store as ss

pd.set_option("display.width", 200)

## 1. Build

`stats_store.build()` loads each dataset with the same steps as the dashboard (corrected coordinates, harmonized
states), runs every test and writes the tables:

| Table | Page | Settings covered |
|---|---|---|
| `missingness` | Data Quality | each dataset × each column with gaps × brewery type, country, state, city |
| `spearman` | Geographic Analysis (capitals) | each analysis dataset × state |
| `type_association` | Brewery Types | each dataset × level × minimum records 5–100 |
| `type_profile_states`, `type_profile_mix`, `type_profile_silhouette` | Brewery Types | each dataset × country × minimum records 5–100 |
| `type_colocation` | Brewery Types | each dataset × country with ≥ 50 geocoded records × same-point option |
| `inputs` | — | SHA-256 of every input file |

In [2]:
rows = ss.build()
pd.Series(rows, name="rows")

missingness                   88
spearman                     412
type_association             864
type_profile_states        14413
type_profile_mix           14744
type_profile_silhouette     3617
type_colocation              426
inputs                         5
Name: rows, dtype: int64

## 2. A look at the stored results

The chi-square test of brewery type against place, at the page's default minimum of 30 records per place, and the
same-type neighbour test for the United States.

In [3]:
with sqlite3.connect(ss.DB) as con:
    T = {name: pd.read_sql_query(f"SELECT * FROM {name}", con)
         for name in ["missingness", "spearman", "type_association", "type_profile_states", "type_profile_mix",
                      "type_profile_silhouette", "type_colocation", "inputs"]}
ss.type_association(T["type_association"], "raw", 30).round(4)

,level,places_tested,records,chi2,dof,p_value,cramers_v
0,country,16,11914,11115.3081,180,0.0,0.2788
1,state,76,11155,14193.6219,900,0.0,0.3256
2,city,22,1052,1544.1682,210,0.0,0.3831


In [4]:
ss.type_colocation(T["type_colocation"], "all_records", "United States", False).round(3)

,brewery_type,records,observed_%,expected_%,p_value,ratio
0,all types,6036,46.123,41.105,0.005,1.122
1,micro,3186,57.815,54.082,0.005,1.069
2,brewpub,1926,41.018,35.050,0.005,1.170
3,closed,340,20.882,18.099,0.180,1.154
4,regional,173,8.092,3.361,0.035,2.408
5,contract,115,9.565,2.255,0.005,4.242
6,planning,108,33.333,6.910,0.005,4.824
7,large,59,8.475,1.303,0.005,6.503
8,taproom,53,20.755,7.130,0.015,2.911
9,proprietor,47,6.383,1.048,0.040,6.092


## 3. Stored equals fresh

A spot check of every table against a fresh computation with SciPy / scikit-learn. The tests repeat it for more
settings.

In [5]:
eq = pd.testing.assert_frame_equal
prepared = {name: ss.load_prepared(name) for name in qc.DATASET_FILES}
raw_stored = ss.load_stored("raw")
eq(ss.missingness(T["missingness"], "raw"),
   pd.DataFrame([t for t in (qc.missingness_association(raw_stored, c, by) for c in ss.missing_targets(raw_stored)
                             for by in qc.CATEGORY_COLUMNS) if t]))
eq(ss.type_association(T["type_association"], "raw", 30),
   pd.DataFrame([dc.type_association(prepared["raw"], lv, min_records=30) for lv in ss.LEVELS]))
for stored, fresh in zip(ss.type_profiles(T["type_profile_states"], T["type_profile_mix"], T["type_profile_silhouette"],
                                          "raw", "United States", 30),
                         dc.type_profiles(prepared["raw"], country="United States", min_records=30)):
    eq(stored, fresh, check_dtype=False, check_index_type=False)  # SciPy labels groups as int32
us = prepared["all_records"][prepared["all_records"]["country"] == "United States"]
eq(ss.type_colocation(T["type_colocation"], "all_records", "United States", True),
   dc.type_colocation(us, permutations=ss.COLOCATION_PERMUTATIONS, exclude_same_point=True))
ref = oc.load_reference()
d = oc.flag_outliers(oc.add_origins(prepared["all_records"], ref), "state", "mean")
eq(oc.central_cities(d, ref)[0], oc.central_cities(d, ref, spearman=ss.spearman(T["spearman"], "all_records"))[0])
print("stored tables match a fresh computation")

stored tables match a fresh computation
